# Cvičenie 5 - Convolutional neural network

Na tomto cvičení si vyskúšate, ako fungujú a ako sa implementujú **konvolučné neurónové siete**. Budete pri tom vychádzať z poznatkov o základoch fungovania "jednoduchých" neurónových sietí z predchádzajúceho cvičenia a rozšírite ich o poznatky o CNN. Opäť budete mať priestor na vlastnú implementáciu modelovej CNN.

## Ciele

1. Oboznámiť sa s konceptom konvolučných neurónových sietí
2. Krok po kroku si predstaviť proces implementácie CNN
3. Získané znalosti aplikovať na už implementovanú sieť z predchádzajúceho cvičenia
4. Navrhnúť, implementovať a otestovať vlastnú CNN sieť

## Zadanie

Na upravenej vyriešenej úlohe z predchádzajúceho cvičenia si naštudujte, ako pracujú a ako sa implementujú konvolučné neurónové siete v rámci Pytorch. Následne implementujte vlastnú konvolučnú sieť, určenú na riešenie rovnakého problému ako v praktickej časti predchádzajúcej úlohy, natrénujte a otestujte ju.

___

## Krok 1 - motivácia pre konvolučné neurónové siete

**Konvolučné neurónové siete** (CNN), sú rozšírením jednoduchých feed-forward neurónových sietí, používajúcim sa najmä v oblasti strojového videnia. Základom CNN sú **konvolučné vrstvy**, ktoré vykonávajú konvolúciu. Ide o proces spracovávania obrazových dát s cieľom extrahovania základných vlastností obrázka (napr. hrany, textúry, tvary), čo umožňuje sieti vykonávať presnejšie predikcie nad dátami.

Typická architektúra CNN, ktorú budete na tomto cvičení implementovať, vyzerá nasledovne:

<img src="./Images/cnn.png" style="width: 900px;"/>

CNN teda obsahjú:


1) **Konvolučné bloky** - základné stavebné bloky CNN, skladajúce sa z: <br/>
    1.1) **Konvolučná vrstva** - vrstva vykonávajúca konvolúciu <br/>
      - Extrahuje z obrázka jeho **vlastnosti**  <br/>
    1.2) **ReLU** - aktivačná funkcia<br/> 
    1.3) **Pooling vrstva** - redukuje veľkosť dát <br/>
      - Napríklad **Max Pooling**
2) **Výstupná vrstva** - lineárna vrstva dostávajúca dáta spracované na vstup, ktorej výstupom je samotná klasifikácia

___

## Krok 2 - riešená vzorová úloha 

V tejto polovici cvičenia si najprv pozorne preštudujte vyriešenú vzorovú úlohu. Konkrétne ide o implementáciu neurónovej siete určenej na klasifikáciu rukou písaných číslic.
V tejto časti cvičenia najprv uvidíte, ako je možné implementovať jednoduchú CNN. Ukážkový príklad vychádza zo vzorovej úlohy predchádzajúceho cvičenia, ktorú modifikuje na implementáciu CNN.

### 2.1 Importy

Naimportujeme potrebné knižnice.

##### *Spustiteľná ukážka*

In [ ]:
import torch # pytorch
import torch.nn as nn # pytorch neural network module
import torch.nn.functional as F # pytorch neural network functional module
import torchvision # pytorch computer vision module
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
from tqdm import tqdm
from datetime import timedelta
import time

___

### 2.2 CPU / GPU ?

Tento krok je rovnaký ako v predchádzajúcom cvičení. Ak vaše zariadenie podporuje GPU akceleráciu, použijeme ju pre urýchlenie trénovania/testovania siete.

##### *Spustiteľná ukážka*

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

___

### 2.3 Vytvorenie Dataset-u, Dataloader-u

Keďže vychádzame z implementácie siete z minulého cvičenia, budeme používať aj rovnké dáta. T.j. dataset MNIST z balíka *torchvision*. Načítame ho a vytvoríme z neho príslušné DataLoader-y, pre trénovacie aj testovacie dáta.

##### *Spustiteľná ukážka*

In [ ]:
batch_size = 100 

# dataset with training data
train_dataset = torchvision.datasets.MNIST(
    root='./Data', 
    download=True,
    train=True, # 
    transform=transforms.ToTensor(),
)

# dataset with testing data
test_dataset = torchvision.datasets.MNIST(
    root='./Data',
    train=False,
    transform=transforms.ToTensor()
)

# dataloader for training data
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

# dataloader for testing data
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)

___

### 2.4 Vytvorenie konvolučnej neurónovej siete

V tomto kroku budeme implementovať samotnú neurónovú sieť. Na rozdiel od predchádzajúceho cvičenia však nepôjde iba o jednoduchý FF model, ale konvolučnú sieť. Model bude teda porovnateľne komplexnejší a bude obsahovať viacero parametrov. Implementujeme ho v nasledujúcej štruktúre:

1) **Dva konvolučné bloky** - použijeme *Sqeuential* vrstvu na zaobalenie <br/>
    1.1) **Konvolučná vrstva** - vykonáva konvolúciu, podľa kernelu nami definovanej veľkosti. <br/>
      - *Pozn*: je potrebné správne definovať počet vstupných kanálov - v našom prípade 1, lebo pracujeme s grayscale dátami (RGB dáta vyžadujú 3 kanály)
    1.2) **ReLU** - aktivačná funkcia<br/> 
    1.3) **Pooling vrstva** - redukuje veľkosť dát<br/>
      - použijeme **Max Pooling**, redukujúci obrázok o polovicu
2) **Výstupná vrstva** - lineárna vrstva o veľkosti 10 neurónov
      - vykonáva klasifikáciu
      
##### *Spustiteľná ukážka*

In [ ]:
# hyperparameters of the network
input_size = 28 * 28 # size of the input layer, derived from size of the MNIST images (28x28 pixels)
num_classes = 10 # number of classes in the classification (10 digits)

# CNN specific hyperparameters
in_channels = 1 # number of input channels (1 because our data is grayscale)
kernel_size = 5 # size of the convolution kernel         
stride = 1 # convolution stride
padding = 2 # size of zero-padding added to input

class CNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNN, self).__init__()

        self.conv1 = nn.Sequential(         
            nn.Conv2d(
                in_channels= in_channels, # color channels of our images             
                out_channels= 16, # number of output channels (chosen by us)        
                kernel_size= kernel_size, # the size of our convolution kernel             
                stride= stride, # stride of the kernel, generally 1
                padding= padding, # size of 0-padding
            ),                              
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=2) #pooling layer, reduces size of data by half
        )
        self.conv2 = nn.Sequential(         
            nn.Conv2d(
                in_channels=16, # number of input channels need to be the same output channels of first convolution block     
                out_channels=32,            
                kernel_size= kernel_size,              
                stride= stride,                   
                padding= padding,                  
            ),  
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=2)             
        )

        self.fc_output = nn.Linear(32 * 7 * 7, num_classes) # linear output layer
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        # flatten the output of conv2 to (batch_size, 32 * 7 * 7)
        x = x.view(x.size(0), -1)       
        output = self.fc_output(x)
        return output
    
model = CNN(input_size,  num_classes).to(device) # initialize a new network with our hyperparameters (on our specified devide, to utilize GPU if available)

print(model)

___

### 2.5 Trénovanie siete


Keď sme si zadefinovali a vytvorili model siete, ďalším krokom je samozrejme jej trénovanie na trénovacích dátach. Princípy trénovnia v slučke po epochách sú vám už známe z predchádzajúceho cvičenia. Použijeme teda rovnaký postup.

Pre zopakovanie, proces trénovacej slučky je nasledovný:
1) Zvoliť si chybovú funkciu
2) Zvoliť si optimizátor
3) Pre každú epochu trénovania:<br/>
    3.1) Pre každú dávku dát (nami určenej veľkosti):<br/>
        3.1.1) Prípadne upraviť tvar dát<br/>
        3.1.2) Vykonať dopredný prechod<br/>
        3.1.3) Vynulovať gradienty optimizátora<br/>
        3.1.4) Vypočítať chybu<br/>
        3.1.5) Vykonať spätný prechod<br/>
        3.1.6) Upraviť váhy v sieti pomocou optimizátora<br/>
        
Nasledujúci kód je teda implementáciou hore uvedného postupu a vykonáva trénovaciu slučku siete.

##### *Spustiteľná ukážka*

In [ ]:
model = CNN(input_size, num_classes).to(device)

# training hyperparamters
num_epochs = 10
learning_rate = 0.001

criterion = nn.CrossEntropyLoss() # the loss function we'll use for training
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) # the optimizing algorithm

start_time = time.time()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} training progress:")

    for i, (images, labels) in enumerate(tqdm(train_loader)): 
        labels = labels.to(device)
        images = images.to(device)
        
        outputs = model(images) # forward pass - do a prediction on our batch

        optimizer.zero_grad() # reset the gradients (so that they don't add up between batches)
        
        loss = criterion(outputs, labels) # calculate loss - the difference between predicted and actual labels
        loss.backward() # backward pass - backpropagate the loss, calculating gradients for neurons
        
        optimizer.step() # let the optimiser adjust weights based on calculated gradients - to minimise loss
    
    if epoch + 1 == num_epochs: 
        print (f'\nTraining took {str(timedelta(seconds=time.time() - start_time))}, loss after all epochs: {loss.item():.5f}\n')    
    else:
        print (f'Loss at end of epoch {epoch+1}: {loss.item():.5f}\n')

___

### 2.6 Testovanie siete

Sieť už máme natrénovanú pomocou trénovacej slučky, je teda zrejmé, že ďalším krokom je validácia jej presnosti. Ukážeme si to na rovnakom postupe ako na predchádzajúcom cvičení, t.j. vykonanie predpovedí na testovacích dátach a vypočítame, koľkým percentám z nich sieť priradila správnu triedu.

##### *Spustiteľná ukážka*

In [ ]:
with torch.no_grad(): # don't track gradients (to reduce memory usage)   
    correct_count = 0
    total_count = 0

    # for each batch, make predictions and calculate how many of them were correct
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        
        total_count += labels.size(0)
        correct_count += (predicted_labels == labels).sum()
        
accuracy = 100. * correct_count / total_count
print (f'Measured accuracy: {accuracy:0.3f}% \n')

Môžete vidieť, že takto definovaný model naozaj dosiahol v porovnaní s jednoduchým FF modelom z minulého cvičenia vyššiu hodnotu presnosti na rovnakých dátach. Je teda zrejmé, že konvolučné neurónové siete sú naozaj vhodným modelom pre klasifikáciu obrázkových dát.

___

## Krok 3 - implementácia vlastnej CNN siete

Po preštudovaní si hore uvedného príkladu by ste mali mať dostatok vedomostí o základných konceptoch CNN a práce s nimi. V tejto časti cvičenia si získané vedomosti môžete vyskúšať na praktickom príklade. Rovnako ako v predchádzajúcom cvičení, implementujte krok po kroku vlastnú CNN, ktorá bude vykonávať predikciu na **CIFAR10** datasete.


### Úloha č.1:

Naimportujte si potrebné knižnice.

In [ ]:
# your code goes here

### Úloha č.2:

Implementujte overenie toho, či vaše zariadenia obsahuje CUDA-enabled GPU. Ak áno, použite ho pri práci s vytváranou sieťou.

In [ ]:
# your code goes here

### Úloha  č.3:

Získajte dataset **CIFAR10** ponúkaný balíkom torchvision a implementujte DataLoader, ktorý obaľuje jeho dáta.

Dokumentácia k datasetu: https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10

In [ ]:
# your code goes here

### Úloha č.4:

Na základe vzorovej úlohy implementujte model CNN, ktorý obsahuje aspoň jeden konvlučný blok. Experimentujte s rôznymi kombináciami vrstiev.

**Nezabudnite správne určiť hyperparametre siete (na veľkosti vstupných obrázkov a počtu tried)**

In [ ]:
# correctly assign this hyperparameter according to the dataset, add ones for convolution layers
num_classes = 

class MyCNN(nn.Module):
    def __init__(self, ____): # add input parameters
        super(MyCNN, self).__init__()

        # define your layers and convolution blocks  

        self.conv1 = nn.Sequential(         
            nn.Conv2d(
                in_channels= _          
                out_channels= _,
                kernel_size= _,         
                stride= _,
                padding= _,
            ),                              
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=_)
        )
        
        # add more convolution blocks if you want
        
        # add the classification layer
        self.fc_output = nn.Linear(_, _)
        
    def forward(self, x):
        # define your forward pass
        
        return out 

### Úloha č.5:

Natrénujte vašu sieť pomocou už známej trénovacej slučky. Môžte experimentovať napr. s výberom rôzych optimilzátorov alebo počtom epoch.

In [ ]:
# create your model (MyCNN class)

# assign training hyperparameters

# choose what loss function and optimiser you want to use


for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(tqdm(train_loader)): 
        # 1. reshape data (if necessary)
        
        # 2. make prediction on batch (forward pass)
        
        # 3. calculate loss, do backward pass and make an optimiser step

### Úloha  č.6:

Keď máte sieť natrénovanú, otestujte ju. Testovanie vykonávajte pomocou už známeho postupu (pomer správnych predikcií), prípadne môžte zvoliť inú, vlastnú metriku.

In [ ]:
# your training loop goes here